In [ ]:
# Installer les dépendances indiquées par le projet.
%pip install -r requirements.txt

In [ ]:
# Bibliothèques utilisées pour collecter et organiser les documents.
import praw
import pandas as pd
import urllib.request
import urllib.parse
import xmltodict
import ssl
import certifi

from datetime import datetime, timezone

# Classes du projet.
from classes.Document import RedditDocument, ArxivDocument
from classes.Corpus import Corpus
from factories.DocumentFactory import DocumentFactory

In [ ]:
from pathlib import Path
from classes.Corpus import Corpus

# Choisir l'origine des données.
mode = input(
    "1 : charger le corpus sauvegardé\n"
    "2 : collecter de nouveaux documents\n"
    "Votre choix : "
).strip()

if mode not in {"1", "2"}:
    raise ValueError("Choisissez 1 ou 2.")

corpus = Corpus("Animaux")

if mode == "1":
    chemin = Path("corpus_objets.csv")

    if not chemin.is_file():
        raise FileNotFoundError(
            "Aucun corpus sauvegardé. Relancez cette cellule "
            "et choisissez 2 pour collecter."
        )

    corpus.load(chemin)
    print("Corpus chargé :", corpus)
else:
    print("Mode collecte sélectionné.")

In [ ]:
if mode == "2":
    from collecte import collecter_documents

    # Connexion Reddit.
    reddit = praw.Reddit("projet")

    # Collecter les textes et conserver leurs métadonnées.
    docs, donnees_documents, nb_reddit, nb_arxiv = collecter_documents(
        reddit=reddit,
        theme_reddit="Cats",
        theme_arxiv="animal",
        taille_reddit=100,
        taille_arxiv=100,
    )

    # Afficher un aperçu court.
    for i, texte in enumerate(docs[:3]):
        print(f"\nDocument {i} : {texte[:200]}...")

In [ ]:
if mode == "2":

    import pandas as pd

    # La collecte ajoute d'abord Reddit, puis arXiv.
    sources = ["reddit"] * nb_reddit + ["arxiv"] * nb_arxiv

    df = pd.DataFrame({
        "id": range(len(docs)),
        "texte": docs,
        "source": sources
    })

    df.to_csv("corpus.csv", sep="\t", index=False, encoding="utf-8")

    print(f"Sauvegarde terminée : {len(df)} documents.")
    print(df["source"].value_counts())

In [ ]:
#Réalise la première sauvegarde des textes des deux sources.
if mode == "2":
    df_charge = pd.read_csv("corpus.csv", sep="\t", encoding="utf-8")

    assert df_charge.equals(df), "Le tableau rechargé est différent !"

    print(f"Rechargement vérifié : {len(df_charge)} documents identiques.")

In [ ]:
# Compter les mots et les phrases après chargement ou collecte.
print("Nombre de documents :", len(corpus.documents))

for identifiant, document in corpus.documents.items():
    texte = document.texte

    nb_mots = len(texte.split())
    nb_phrases = len([
        phrase
        for phrase in texte.split(".")
        if phrase.strip()
    ])

    print(
        f"Document {identifiant} : "
        f"{nb_mots} mots, {nb_phrases} phrases"
    )

In [ ]:
if mode == "2":
    nb_avant = len(df)

    df = df[df["texte"].str.len() >= 100].copy()
    df = df.reset_index(drop=True)

    # Réattribuer des identifiants consécutifs.
    df["id"] = range(len(df))

    # Mettre à jour la liste des textes.
    docs = df["texte"].tolist()

    print("Documents supprimés :", nb_avant - len(df))
    print("Documents conservés :", len(docs))

In [ ]:
# Sauvegarder les textes filtrés après une nouvelle collecte.
if mode == "2":
    df.to_csv(
        "corpus.csv",
        sep="\t",
        index=False,
        encoding="utf-8",
    )

    print("Corpus filtré sauvegardé.")

In [ ]:
if mode == "2":
    # Construire le corpus avec les docs.
    corpus = Corpus("Animaux")

    corpus.clear()

    for donnees in donnees_documents:
        if len(donnees["texte"]) < 100:
            continue

        attributs = donnees.copy()
        source = attributs.pop("source")

        document = DocumentFactory.create(
            source=source,
            **attributs
        )

        corpus.add(document)

    assert len(corpus.documents) == len(df)

    assert [
        document.texte
        for document in corpus.documents.values()
    ] == df["texte"].tolist()

    print(corpus)

In [ ]:

# Afficher les documents réels par titre et par date.
print("Par titre :")
corpus.afficher_par_titre(5)

print("\nPar date, les plus récents d'abord :")
corpus.afficher_par_date(5)

In [ ]:
# Vérifier le polymorphisme et afficher la source de chaque doc.
if mode == "2":
    for identifiant, document in corpus.documents.items():
        assert isinstance(document, (RedditDocument, ArxivDocument))
        print(identifiant, document.getType(), ":", document)

    print("\nNombre de documents par source :")
    print(pd.Series([
        document.getType()
        for document in corpus.documents.values()
    ]).value_counts())

In [ ]:
# Mémoriser les données avant de recharger le corpus unique.
if mode == "2":

    avant = [
        (type(document).__name__, vars(document).copy())
        for document in corpus.documents.values()
    ]

    corpus.save("corpus_objets.csv")
    corpus.load("corpus_objets.csv")

    apres = [
        (type(document).__name__, vars(document).copy())
        for document in corpus.documents.values()
    ]

    assert avant == apres

    print("Sauvegarde vérifiée : types et attributs identiques.")
    print(corpus)

In [ ]:
# Vérifier qu'il y a pas de deoublons.
if mode == "2":
    autre_reference = Corpus("Autre nom")

    assert autre_reference is corpus

    print("Singleton vérifié :", autre_reference is corpus)

In [ ]:
# Voir les stats d'un auteur réel du corpus des docs.
print("Auteurs disponibles :")
for nom in corpus.authors:
    print("-", nom)

nom = input("\nNom de l'auteur : ").strip()
auteur = corpus.authors.get(nom)

if auteur is None:
    print("Auteur introuvable.")
else:
    taille_moyenne = (
        sum(len(document.texte) for document in auteur.production.values())
        / auteur.nb_docs
        if auteur.nb_docs
        else 0
    )

    print("Nombre de documents :", auteur.nb_docs)
    print(
        "Taille moyenne :",
        round(taille_moyenne, 1),
        "caractères"
    )

In [ ]:
import importlib
import classes.Corpus as corpus_module

importlib.reload(corpus_module)

corpus = corpus_module.Corpus("Animaux")
corpus.load("corpus_objets.csv") 
print(len(corpus.documents), "document(s)")
print(corpus.search("cat")[:3])

In [ ]:
import importlib
import classes.SearchEngine as se_module

importlib.reload(se_module)

moteur = se_module.SearchEngine(corpus)
print(moteur.mat_TFxIDF.shape)
moteur.search("cat", n=5)